# 🧠 Image Segmentation Pipeline - Demo Notebook

End-to-end walkthrough of the semantic segmentation pipeline:
1. Data loading & visualization
2. Model instantiation
3. Training (mini run)
4. Evaluation & metrics
5. Post-processing
6. Visualization

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
from src.models import get_model, list_models
from src.datasets.segmentation_dataset import SyntheticSegmentationDataset, create_dataloaders
from src.augmentation.transforms import get_augmentation_pipeline, Mixup
from src.training.trainer import Trainer
from src.evaluation.metrics import mean_iou, pixel_accuracy, dice_coefficient
from src.evaluation.evaluator import Evaluator
from src.postprocessing.postprocess import PostProcessor
from src.utils.config import load_config

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Available models: {list_models()}')

## 1. Synthetic Dataset

In [ ]:
# Create synthetic dataset
dataset = SyntheticSegmentationDataset(
    num_samples=100,
    image_size=256,
    num_classes=10,
    seed=42
)
print(f'Dataset size: {len(dataset)}')

# Visualize samples
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    sample = dataset[i]
    img = sample['image'].permute(1, 2, 0).numpy()
    mask = sample['mask'].numpy()
    axes[0, i].imshow(img)
    axes[0, i].set_title(f'Image {i+1}')
    axes[0, i].axis('off')
    axes[1, i].imshow(mask, cmap='tab20')
    axes[1, i].set_title(f'Mask {i+1}')
    axes[1, i].axis('off')
plt.suptitle('Synthetic Training Samples', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Model Architecture

In [ ]:
# U-Net
unet = get_model('unet', num_classes=10, pretrained=True)
x = torch.randn(1, 3, 256, 256)
out = unet(x)
print(f'U-Net: input={x.shape} -> output={out.shape}')
print(f'U-Net parameters: {sum(p.numel() for p in unet.parameters()):,}')

# DeepLabV3
deeplab = get_model('deeplabv3', num_classes=10, pretrained=True)
out2 = deeplab(x)
print(f'DeepLabV3: input={x.shape} -> output={out2.shape}')
print(f'DeepLabV3 parameters: {sum(p.numel() for p in deeplab.parameters()):,}')

## 3. Quick Training Run

In [ ]:
# Load config and reduce for demo
config = load_config('../configs/default.yaml')
config['training']['epochs'] = 3
config['training']['batch_size'] = 4
config['model']['num_classes'] = 10
config['dataset']['synthetic']['num_samples'] = 100
config['logging']['tensorboard'] = False
config['checkpoint']['save_dir'] = '/tmp/demo_checkpoints'
config['logging']['log_dir'] = '/tmp/demo_logs'

# Create model and data
model = get_model('unet', num_classes=10, pretrained=True)
transform = get_augmentation_pipeline('basic', config.get('augmentation', {}))
train_loader, val_loader = create_dataloaders(config, transform)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches: {len(val_loader)}')

# Train
trainer = Trainer(model, config, device)
history = trainer.train(train_loader, val_loader)

## 4. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs = range(1, len(history['train_losses']) + 1)

axes[0].plot(epochs, history['train_losses'], 'b-', label='Train')
axes[0].plot(epochs, history['val_losses'], 'r-', label='Val')
axes[0].set_title('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

train_miou = [m['miou'] for m in history['train_metrics']]
val_miou = [m['miou'] for m in history['val_metrics']]
axes[1].plot(epochs, train_miou, 'b-', label='Train')
axes[1].plot(epochs, val_miou, 'r-', label='Val')
axes[1].set_title('Mean IoU')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

train_acc = [m['accuracy'] for m in history['train_metrics']]
val_acc = [m['accuracy'] for m in history['val_metrics']]
axes[2].plot(epochs, train_acc, 'b-', label='Train')
axes[2].plot(epochs, val_acc, 'r-', label='Val')
axes[2].set_title('Pixel Accuracy')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('Training Curves', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Evaluation & Metrics

In [ ]:
# Quick metric demo
pred = torch.randint(0, 10, (4, 256, 256))
target = torch.randint(0, 10, (4, 256, 256))

print(f'mIoU:           {mean_iou(pred, target, 10):.4f}')
print(f'Pixel Accuracy: {pixel_accuracy(pred, target):.4f}')
print(f'Dice:           {dice_coefficient(pred, target, 10):.4f}')

# Perfect prediction
perfect_pred = target.clone()
print(f'\nPerfect prediction:')
print(f'mIoU:           {mean_iou(perfect_pred, target, 10):.4f}')
print(f'Pixel Accuracy: {pixel_accuracy(perfect_pred, target):.4f}')

## 6. Post-Processing

In [ ]:
# Post-processing demo
processor = PostProcessor({'small_region_threshold': 50, 'morphology': {'kernel_size': 5, 'operations': ['opening', 'closing']}})

# Create a noisy mask
sample = dataset[0]
raw_mask = sample['mask'].numpy()

# Add noise
noisy_mask = raw_mask.copy()
noise = np.random.randint(0, 10, size=noisy_mask.shape).astype(np.uint8)
noise_positions = np.random.random(noisy_mask.shape) < 0.05
noisy_mask[noise_positions] = noise[noise_positions]

# Apply post-processing
processed = processor.process(noisy_mask)
contours = processor.extract_contours(processed)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(noisy_mask, cmap='tab20')
axes[0].set_title(f'Noisy Mask')
axes[0].axis('off')
axes[1].imshow(processed, cmap='tab20')
axes[1].set_title(f'Post-Processed')
axes[1].axis('off')
axes[2].imshow(raw_mask, cmap='tab20')
axes[2].set_title(f'Original (GT)')
axes[2].axis('off')
plt.suptitle(f'Post-Processing Demo ({len(contours)} contours extracted)', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Model Predictions

In [ ]:
# Run predictions
model.eval()
fig, axes = plt.subplots(3, 4, figsize=(16, 12))

for i in range(4):
    sample = dataset[i]
    img = sample['image'].unsqueeze(0).to(device)
    
    with torch.no_grad():
        pred = model(img).argmax(dim=1).squeeze().cpu().numpy()
    
    axes[0, i].imshow(sample['image'].permute(1, 2, 0).numpy())
    axes[0, i].set_title('Input')
    axes[0, i].axis('off')
    axes[1, i].imshow(sample['mask'].numpy(), cmap='tab20', vmin=0, vmax=9)
    axes[1, i].set_title('Ground Truth')
    axes[1, i].axis('off')
    axes[2, i].imshow(pred, cmap='tab20', vmin=0, vmax=9)
    axes[2, i].set_title('Prediction')
    axes[2, i].axis('off')

plt.suptitle('Segmentation Results', fontsize=14)
plt.tight_layout()
plt.show()
print('Demo complete!')